In [ ]:
from google.colab import files
from zipfile import ZipFile
from pathlib import Path
import pandas as pd

uploaded = files.upload()  # Select prepared_datasets.zip
archive = next(name for name in uploaded if name.endswith(".zip"))

with ZipFile(archive) as z:
    z.extractall("/content")

dataset = Path(
    "/content/prepared_datasets/linevul/primevul_test.csv"
)
assert dataset.is_file(), f"Dataset missing: {dataset}"

df = pd.read_csv(dataset)
print("Dataset:", dataset)
print("Rows:", len(df))
print("Label counts:", df["target"].value_counts().to_dict())

Saving prepared_datasets.zip to prepared_datasets (1).zip
Dataset: /content/prepared_datasets/linevul/primevul_test.csv
Rows: 24788
Label counts: {0: 24239, 1: 549}


In [3]:
%pip install -q gdown

from pathlib import Path
import subprocess
import gdown

repo = Path("/content/LineVul")
if not repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/awsm-research/LineVul.git", str(repo)],
        check=True,
    )

checkpoint = (
    repo / "linevul/saved_models/checkpoint-best-f1"
    / "12heads_linevul_model.bin"
)
checkpoint.parent.mkdir(parents=True, exist_ok=True)

if not checkpoint.exists():
    result = gdown.download(
        id="1oodyQqRb9jEcvLMVVKILmu8qHyNwd-zH",
        output=str(checkpoint),
        quiet=False,
    )
    if result is None:
        raise RuntimeError("Checkpoint download failed.")

print("Checkpoint:", checkpoint)
print("Bytes:", checkpoint.stat().st_size)

Downloading...
From (original): https://drive.google.com/uc?id=1oodyQqRb9jEcvLMVVKILmu8qHyNwd-zH
From (redirected): https://drive.google.com/uc?id=1oodyQqRb9jEcvLMVVKILmu8qHyNwd-zH&confirm=t&uuid=e605e6bf-4efa-48a7-99ed-3c5d7ec343dd
To: /content/LineVul/linevul/saved_models/checkpoint-best-f1/12heads_linevul_model.bin
100%|██████████| 997M/997M [00:13<00:00, 73.2MB/s]

Checkpoint: /content/LineVul/linevul/saved_models/checkpoint-best-f1/12heads_linevul_model.bin
Bytes: 997349337


In [4]:
from pathlib import Path

path = Path("/content/LineVul/linevul/linevul_main.py")
source = path.read_text(encoding="utf-8")

old = "    for batch in test_dataloader:"
new = (
    "    for batch in tqdm(test_dataloader, total=len(test_dataloader), "
    'desc="LineVul inference", unit="batch", mininterval=2.0, '
    "dynamic_ncols=True):"
)

if new in source:
    print("Progress bar already installed.")
else:
    assert source.count(old) == 1, "Unexpected source layout; nothing changed."
    assert "from tqdm import tqdm" in source

    patched = source.replace(old, new, 1)
    compile(patched, str(path), "exec")

    backup = path.with_name("linevul_main.before_progress.py")
    if not backup.exists():
        backup.write_bytes(path.read_bytes())

    path.write_text(patched, encoding="utf-8")
    print("Progress bar installed. Rerun the evaluation cell.")

Progress bar installed. Rerun the evaluation cell.


Reviewing PrimeVul on LineVul checkpoint

In [ ]:
%pip install -q captum
%cd /content/LineVul/linevul

!python linevul_main.py \
    --model_name=12heads_linevul_model.bin \
    --output_dir=./saved_models \
    --model_type=roberta \
    --tokenizer_name=microsoft/codebert-base \
    --model_name_or_path=microsoft/codebert-base \
    --do_test \
    --test_data_file=/content/prepared_datasets/linevul/primevul_test.csv \
    --block_size=512 \
    --eval_batch_size=16

/content/LineVul/linevul
10/09/2026 01:52:16 - WARNING - __main__ -   device: cuda, n_gpu: 1
10/09/2026 01:52:16 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/revision/main "HTTP/1.1 200 OK"
10/09/2026 01:52:16 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/revision/main "HTTP/1.1 200 OK"
10/09/2026 01:52:16 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
10/09/2026 01:52:16 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
10/09/2026 01:52:16 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/revision/main "HTTP/1.1 200 OK"
10/09/2026 01:52:16 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/microsoft

Reviewing LLMVul on same LineVul checkpoint

In [ ]:
%cd /content/LineVul/linevul

!python -u linevul_main.py \
    --model_name=12heads_linevul_model.bin \
    --output_dir=./saved_models \
    --model_type=roberta \
    --tokenizer_name=microsoft/codebert-base \
    --model_name_or_path=microsoft/codebert-base \
    --do_test \
    --test_data_file=/content/prepared_datasets/linevul/llmvul_test.csv \
    --block_size=512 \
    --eval_batch_size=16

/content/LineVul/linevul
10/09/2026 03:03:42 - WARNING - __main__ -   device: cuda, n_gpu: 1
10/09/2026 03:03:42 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/revision/main "HTTP/1.1 200 OK"
10/09/2026 03:03:42 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/revision/main "HTTP/1.1 200 OK"
10/09/2026 03:03:42 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
10/09/2026 03:03:42 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
10/09/2026 03:03:43 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/revision/main "HTTP/1.1 200 OK"
10/09/2026 03:03:43 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/microsoft

In [ ]:
%pip install -q "transformers==4.57.1" libclang safetensors pandas scikit-learn tqdm
%cd /content

from google.colab import files
from pathlib import Path

uploaded = files.upload()  # Select the newly downloaded script
name = next(name for name in uploaded if name.endswith(".py"))
script = uploaded[name].decode("utf-8")

assert "PARSE_SKIP_FUNCTION_BODIES" in script, "You selected the old script."

Path("/content/vulberta_test.py").write_text(script, encoding="utf-8")
print("Updated VulBERTa script installed.")

/content


Saving vulberta_test.py to vulberta_test.py
Updated VulBERTa script installed.


In [ ]:
%cd /content

!python -u vulberta_test.py \
    --model_name_or_path=claudios/VulBERTa-MLP-Devign \
    --test_data_file=/content/prepared_datasets/vulberta/primevul_test.csv \
    --output_dir=/content/vulberta_results \
    --block_size=1024 \
    --eval_batch_size=16

/content
I0000 00:00:1791519488.516728   46269 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791519490.848803   46269 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
device: cuda; GPU: Tesla T4
Dataset: /content/prepared_datasets/vulberta/primevul_test.csv; rows: 24788
Label counts: {0: 24239, 1: 549}
Checkpoint: claudios/VulBERTa-MLP-Devign@fecabdee24a3ef6e804d7629d6eed03dc6b4f61e
tokenizer_config.json: 100% 620/620 [00:00<00:00, 4.08MB/s]
tokenization_vulberta.py: 1.39kB [00:00, 2.98MB/s]
tokenizer.json: 2.05MB [00:00, 20.9MB/s]
special_tokens_map.json: 100% 27.0/27.0 [00:00<00:00, 275kB/s]
config.json: 100% 733/733 [00:00<00:00, 6.03MB/s]
model.safetensors: 100% 499M/499M [00:03<00:00, 146MB/s]
Trained classifier load

In [ ]:
%cd /content

!python -u vulberta_test.py \
    --model_name_or_path=claudios/VulBERTa-MLP-Devign \
    --test_data_file=/content/prepared_datasets/vulberta/llmvul_test.csv \
    --output_dir=/content/vulberta_results_v2 \
    --block_size=1024 \
    --eval_batch_size=16 \
    --resume

/content
device: cuda; GPU: Tesla T4
Dataset: /content/prepared_datasets/vulberta/llmvul_test.csv; rows: 18751
Label counts: {0: 17211, 1: 1540}
Checkpoint: claudios/VulBERTa-MLP-Devign@fecabdee24a3ef6e804d7629d6eed03dc6b4f61e
Trained classifier loaded successfully. Starting test.
Completed rows restored: 0
Clang parser fix enabled; completed batches will be saved.
VulBERTa test: 100% 1172/1172 [21:22<00:00,  1.09s/batch]

***** Test results *****
  test_accuracy = 0.6325
  test_f1 = 0.1477
  test_precision = 0.0912
  test_recall = 0.3877
  test_threshold = 0.5000
Saved metrics.json and predictions.csv to /content/vulberta_results_v2/llmvul_test


For my sanity to see if I've messed up the models

In [6]:
%pip install -q gdown captum

from pathlib import Path
import gdown
import pandas as pd
import torch

repo = Path("/content/LineVul/linevul")
checkpoint = repo / "saved_models/checkpoint-best-f1/12heads_linevul_model.bin"
dataset = Path("/content/bigvul_test.csv")

#assert torch.cuda.is_available(), "Enable the GPU runtime."
assert checkpoint.is_file(), "Rerun your earlier LineVul setup/download cell."

if not dataset.exists():
    result = gdown.download(
        id="1h0iFJbc5DGXCXXvvR6dru_Dms_b2zW4V",
        output=str(dataset),
        quiet=False,
    )
    assert result is not None, "BigVul download failed."

df = pd.read_csv(dataset, usecols=["processed_func", "target"])
assert df["processed_func"].notna().all()
assert df["target"].isin([0, 1]).all()

#print("GPU:", torch.cuda.get_device_name(0))
print("Rows:", len(df))
print("Label counts:", df["target"].value_counts().to_dict())
print("Vulnerable percentage:", round(100 * df["target"].mean(), 2))

Path("/content/linevul_bigvul_results/results").mkdir(
    parents=True, exist_ok=True
)

Rows: 18864
Label counts: {0: 17809, 1: 1055}
Vulnerable percentage: 5.59


This is cutting the dataset down just for a spot check

In [8]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("/content/bigvul_test.csv")

sample, _ = train_test_split(
    df,
    train_size=2000,
    stratify=df["target"],
    random_state=42,
)

sample = sample.sort_index()
sample.to_csv("/content/bigvul_test_sample_2000.csv", index=False)

print("Full test rows:", len(df))
print("Sample rows:", len(sample))
print("Sample label counts:", sample["target"].value_counts().to_dict())
print("Full vulnerable proportion:", round(df["target"].mean(), 4))
print("Sample vulnerable proportion:", round(sample["target"].mean(), 4))

Full test rows: 18864
Sample rows: 2000
Sample label counts: {0: 1888, 1: 112}
Full vulnerable proportion: 0.0559
Sample vulnerable proportion: 0.056


In [7]:
from pathlib import Path

Path("/content/linevul_bigvul_cpu_results/results").mkdir(
    parents=True, exist_ok=True
)

%cd /content/linevul_bigvul_cpu_results

!CUDA_VISIBLE_DEVICES="" USE_TF=0 python -u /content/LineVul/linevul/linevul_main.py \
    --model_name=12heads_linevul_model.bin \
    --output_dir=/content/LineVul/linevul/saved_models \
    --model_type=roberta \
    --tokenizer_name=microsoft/codebert-base \
    --model_name_or_path=microsoft/codebert-base \
    --do_test \
    --test_data_file=/content/bigvul_test.csv \
    --block_size=512 \
    --eval_batch_size=4 \
    --write_raw_preds \
    2>&1 | tee /content/linevul_bigvul_cpu_results/test.log

/content/linevul_bigvul_cpu_results
10/09/2026 23:32:04 - WARNING - __main__ -   device: cpu, n_gpu: 0
10/09/2026 23:32:05 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
10/09/2026 23:32:05 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/revision/main "HTTP/1.1 200 OK"
10/09/2026 23:32:05 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/3b0952feddeffad0063f274080e3c23d75e7eb39/config.json "HTTP/1.1 307 Temporary Redirect"
10/09/2026 23:32:05 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/codebert-base/3b0952feddeffad0063f274080e3c23d75e7eb39/config.json?%2Fmicrosoft%2Fcodebert-base%2Fresolve%2F3b0952feddeffad0063f274080e3c23d75e7eb39%2Fconfig.json=&etag=%22cd6f88de92758ae36c1c6e18a03ad288ccb41172%22 "HTTP/1.1 200 OK"
10/09/2026 23:32:05 - INFO - httpx -   HTTP Request: GET https://huggingface.co/ap

Test for the small sample because I'm out of GPU usage

In [9]:
from pathlib import Path

Path("/content/linevul_bigvul_sample_cpu_results/results").mkdir(
    parents=True, exist_ok=True
)

%cd /content/linevul_bigvul_sample_cpu_results

!CUDA_VISIBLE_DEVICES="" USE_TF=0 python -u /content/LineVul/linevul/linevul_main.py \
    --model_name=12heads_linevul_model.bin \
    --output_dir=/content/LineVul/linevul/saved_models \
    --model_type=roberta \
    --tokenizer_name=microsoft/codebert-base \
    --model_name_or_path=microsoft/codebert-base \
    --do_test \
    --test_data_file=/content/bigvul_test_sample_2000.csv \
    --block_size=512 \
    --eval_batch_size=4 \
    --write_raw_preds \
    2>&1 | tee /content/linevul_bigvul_sample_cpu_results/test.log

/content/linevul_bigvul_sample_cpu_results
10/09/2026 23:37:23 - WARNING - __main__ -   device: cpu, n_gpu: 0
10/09/2026 23:37:23 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/revision/main "HTTP/1.1 200 OK"
10/09/2026 23:37:24 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/revision/main "HTTP/1.1 200 OK"
10/09/2026 23:37:24 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
10/09/2026 23:37:24 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
10/09/2026 23:37:24 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/revision/main "HTTP/1.1 200 OK"
10/09/2026 23:37:24 - INFO - httpx -   HTTP Request: HEAD https://hugging